In [4]:
from dotenv import load_dotenv
load_dotenv(r"c:\Users\Admin\hipython\learning\llm\.env", override=True)
import os

## Pinecone 인덱스 생성

In [5]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

INDEX_NAME = "finance-bok"
NAMESPACE  = "bok-ns1"
# 기존 인덱스 없으면 생성
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"{INDEX_NAME} 생성 완료")
else:
    print(f"{INDEX_NAME} 이미 존재함")


finance-bok 생성 완료


In [6]:
# 상태 확인
f_index = pc.Index(INDEX_NAME)
print(f_index .describe_index_stats())

c:\Users\Admin\miniconda3\envs\langchain_rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [12]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()
load_dotenv(override=True)

print(f"총 페이지 수: {len(docs)}")
print(f"\n첫 페이지 내용 (앞 300자):\n{docs[0].page_content[:300]}")
print(f"\n메타데이터: {docs[0].metadata}")

총 페이지 수: 93

첫 페이지 내용 (앞 300자):
경제전망 
 Indigo Book 
2026년 2월 
 
 
 
 
성장 2%대 반등, 부문별 온도차

메타데이터: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-26T16:48:20+09:00', 'author': 'A11', 'moddate': '2026-03-31T15:12:50+09:00', 'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 0, 'page_label': '1'}


## 청킹

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)


In [14]:
splits = splitter.split_documents(docs) #원본 문서의 메타 정보를 가져와 설정

In [15]:
print(f"총 청크 수: {len(splits)}")
print(f"\n첫 번째 청크:\n{splits[0].page_content}")
print(f"\n메타데이터: {splits[0].metadata}")

총 청크 수: 249

첫 번째 청크:
경제전망 
 Indigo Book 
2026년 2월 
 
 
 
 
성장 2%대 반등, 부문별 온도차

메타데이터: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-26T16:48:20+09:00', 'author': 'A11', 'moddate': '2026-03-31T15:12:50+09:00', 'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 0, 'page_label': '1'}


In [16]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

## 업서트

In [17]:
print("업서트 시작...")
vectorstore = PineconeVectorStore.from_documents(
    documents=splits,
    embedding=embedding_model ,
    index_name=INDEX_NAME,
    namespace=NAMESPACE
)
print(f"업서트 완료 — 총 {len(splits)}개 청크")

업서트 시작...
업서트 완료 — 총 249개 청크


In [18]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
print(stats)

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'bok-ns1': {'vector_count': 249}},
 'total_vector_count': 249,
 'vector_type': 'dense'}


## Retriever 생성

In [19]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "namespace": NAMESPACE}
)

## 검색 테스트

In [20]:
queries = [
    "2026년 경제성장률 전망은?",
    "물가 상승률 전망은?",
    "고용 시장 전망은?"
]

for query in queries:
    print(f"[ 질문 ] {query}")
    results = retriever.invoke(query)
    for i, r in enumerate(results):
        print(f"  {i+1}. (p.{r.metadata.get('page', '')}) {r.page_content[:100]}")
    print()

[ 질문 ] 2026년 경제성장률 전망은?
  1. (p.43.0) 30 
 
<경제성장 전망1)> 
(전년동기대비, %) 
  2024 2025 2026e) 2027e) 
연간 상반 하반 연간 상반 하반 연간 연간 
GDP 성장률 2.0 0.3 
  2. (p.35.0) 22 
 
2. 거시경제 전망 
 
 
 
 
경제성장 
 
2-1. 금년중 국내경제는 美관세 영향, 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한
  3. (p.6.0) < 요약 1/8 > 
 
  
경제전망 요약 
 올해 우리 경제는 美관세 영향과 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한 세계경제 흐름 등에

[ 질문 ] 물가 상승률 전망은?
  1. (p.53.0) 1.9% 대비 0.1%p 높아졌다. 소비자물가 상승률의 경우에도 11월 전망1.9% 대비 0.1%p 높
아진 2.0%로 나타났다. 
 
시장의 국내 성장 및 물가 전망 모두 올해 
  2. (p.53.0) 40 
 
3. 전망의 리스크 평가 
    
주요 리스크 요인 
 
3-1. 향후 성장 전망경로에는 반도체 경기, 글로벌 통상환경, 국제금융시장 등과 관련
한 불확실성이 크며, 
  3. (p.11.0) < 요약 6/8 > 
6  
  
전망의 리스크 
 
 향후 성장 전망경로에는 반도체 경기, 글로벌 통상환경, 국제금융시장 등과 관련한 불확
실성이 크며, 물가의 경우 유가, 환

[ 질문 ] 고용 시장 전망은?
  1. (p.10.0) ▪ 상품수지는 반도체가격의 큰 폭 상승 등으로 흑자규모가 크게 늘어날 전망이다. 
서비스수지는 경기회복에 따른 산업서비스특허사용료 등 수요 증가, 디지털서비스플랫폼 구
독료 등 지
  2. (p.35.0) 22 
 
2. 거시경제 전망 
 
 
 
 
경제성장 
 
2-1. 금년중 국내경제는 美관세 영향, 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한
  3. (p.53.0) 40 


## RAG 체인 생성

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import ChatOpenAI

llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("""
당신은 한국은행 경제전망 보고서를 기반으로 답변하는 금융 전문 어시스턴트입니다.
아래 참고 문서를 바탕으로 질문에 정확하게 답하세요.
문서에 없는 내용은 "보고서에서 확인되지 않습니다"라고 답하세요.

[참고문서]
{context}

[질문]
{question}

한글로 간결하고 정확하게 답변하세요.
""")

rag_chain = (
    RunnableParallel(
        context=retriever,
        question=RunnablePassthrough()
    )
    | prompt
    | llm
    | parser
)

## RAG 테스트

In [22]:
questions = [
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "소비자물가 상승률은 어떻게 전망하나요?",
    "수출 전망은 어떻게 되나요?"
]

for q in questions:
    print(f"[ Q ] {q}")
    answer = rag_chain.invoke(q)
    print(f"[ A ] {answer}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ A ] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 소비자물가 상승률은 어떻게 전망하나요?
[ A ] 소비자물가 상승률은 2.0%로 전망되고 있습니다. 이는 11월 전망인 1.9% 대비 0.1%p 높아진 수치입니다.

[ Q ] 수출 전망은 어떻게 되나요?
[ A ] 수출은 견조한 흐름을 이어갈 것으로 전망됩니다. 2024년 수출은 6,836억 달러에서 시작하여 2025년에는 7,093억 달러, 2026년에는 7,952억 달러로 증가할 것으로 예상됩니다.



In [33]:
questions = [
    "그래프에서 GDP 전망경로는 어떻게 되나요?",
    "국내 설비투자 성장률 전망은 어떻게 되나요?",
    "몇 분기에 소비회복세가 확대될 것인가요?"
]


for q in questions:
    print(f"[ Q ] {q}")
    answer = rag_chain.invoke(q)
    print(f"[ A ] {answer}")
    print()

[ Q ] 그래프에서 GDP 전망경로는 어떻게 되나요?
[ A ] 2025년 1분기 GDP 성장률은 0.0%, 2분기 0.6%, 3분기 1.8%, 4분기 1.5%로 예상되며, 2026년 1분기 2.7%, 2분기 2.2%, 3분기 1.3%, 4분기 2.0%로 전망됩니다.

[ Q ] 국내 설비투자 성장률 전망은 어떻게 되나요?
[ A ] 국내 설비투자 성장률 전망은 2024년 2.4%, 2025년 2.0%로 예상됩니다.

[ Q ] 몇 분기에 소비회복세가 확대될 것인가요?
[ A ] 보고서에서 확인되지 않습니다.



| 확인 항목 | 좋은 경우 | 나쁜 경우 |
| --- | --- | --- |
| 청크 수 | k=3개 모두 반환 | 0개 또는 1개 |
| 관련성 | 질문과 직접 관련된 내용 | 전혀 다른 주제 |
| 출처 페이지 | 특정 페이지에 집중 | 무작위 페이지 |
| 내용 완결성 | 문장이 끊기지 않음 | 중간에 잘림 |

## 답변 품질 확인

### RAG 답변 vs 일반 LLM 답변 비교

In [23]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm_base = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

In [24]:
questions = [
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "소비자물가 상승률은 어떻게 전망하나요?",
    "수출 전망은 어떻게 되나요?"
]

In [25]:
for q in questions:
    print(f"[ Q ] {q}")

    # 일반 LLM (RAG 없음)
    base_answer = parser.invoke(llm_base.invoke(q))
    print(f"[ 일반 LLM ] {base_answer[:150]}")

    # RAG 기반 LLM
    rag_answer = rag_chain.invoke(q)
    print(f"[ RAG 답변 ] {rag_answer[:150]}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ 일반 LLM ] 2026년 GDP 성장률 전망치는 여러 경제 기관과 연구소에 따라 다를 수 있습니다. 일반적으로 이러한 전망치는 경제 상황, 정책 변화, 글로벌 경제 동향 등에 따라 변동할 수 있습니다. 특정 국가나 지역에 대한 구체적인 전망치를 원하신다면, 해당 국가의 중앙은행, 정
[ RAG 답변 ] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 소비자물가 상승률은 어떻게 전망하나요?
[ 일반 LLM ] 소비자물가 상승률 전망은 여러 요인에 따라 달라질 수 있습니다. 일반적으로 경제 성장률, 통화 정책, 원자재 가격, 공급망 문제, 그리고 글로벌 경제 상황 등이 주요한 영향을 미칩니다. 

2023년에는 많은 국가에서 인플레이션이 높은 수준을 유지하고 있으며, 이는 에
[ RAG 답변 ] 소비자물가 상승률은 2.0%로 전망됩니다. 이는 11월 전망인 1.9% 대비 0.1%p 높아진 수치입니다.

[ Q ] 수출 전망은 어떻게 되나요?
[ 일반 LLM ] 수출 전망은 여러 요인에 따라 달라질 수 있습니다. 일반적으로 경제 성장률, 글로벌 수요, 환율 변동, 무역 정책, 그리고 특정 산업의 경쟁력 등이 중요한 요소로 작용합니다. 

2023년의 경우, 세계 경제의 회복세와 주요 국가들의 수요 증가가 긍정적인 영향을 미칠 
[ RAG 답변 ] 수출은 견조한 흐름을 이어갈 것으로 전망됩니다.



| 답변 유형 | 판단 | 예시 |
| --- | --- | --- |
| 수치가 정확하게 포함됨 | 우수 | "2026년 GDP 성장률은 2.0%입니다" |
| 방향성만 맞고 수치 없음 | 보통 | "성장세가 회복될 것으로 전망됩니다" |
| 보고서에 없는 내용 생성 | 불량 | "3.5% 성장이 예상됩니다" (보고서와 다른 수치) |
| 모른다고 정직하게 답변 | 정상 | "보고서에서 확인되지 않습니다" |

## 종합 품질 점검

In [34]:
test_questions = [
    # 수치 확인용
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "2026년 소비자물가 상승률 전망치는?",
    "2026년 수출 전망 금액은 얼마인가요?",

    # 범위 밖 질문 (환각 테스트)
    "2030년 경제성장률 전망은?",
    "미국 연방준비제도의 금리 결정 일정은?",

    # 맥락 이해 확인
    "성장률이 반등하는 주요 원인은?",
    "부문별 온도차가 발생하는 이유는?"
]

In [35]:
for q in test_questions:
    print(f"[ Q ] {q}")

    # RAG 기반 LLM
    rag_answer = rag_chain.invoke(q)
    print(f"[ RAG 답변 ] {rag_answer[:300]}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ RAG 답변 ] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 2026년 소비자물가 상승률 전망치는?
[ RAG 답변 ] 2026년 소비자물가 상승률 전망치는 2.0%입니다.

[ Q ] 2026년 수출 전망 금액은 얼마인가요?
[ RAG 답변 ] 2026년 수출 전망 금액은 7,952억 달러입니다.

[ Q ] 2030년 경제성장률 전망은?
[ RAG 답변 ] 보고서에서 확인되지 않습니다.

[ Q ] 미국 연방준비제도의 금리 결정 일정은?
[ RAG 답변 ] 보고서에서 확인되지 않습니다.

[ Q ] 성장률이 반등하는 주요 원인은?
[ RAG 답변 ] 보고서에서 확인되지 않습니다.

[ Q ] 부문별 온도차가 발생하는 이유는?
[ RAG 답변 ] 부문별 온도차는 주로 IT 대기업이 주도하는 K자형 회복국면에서 발생하며, 이는 일부 부문(예: 반도체)에서의 성장과 여타 부문 간의 성장 차별화로 인해 나타납니다. 이러한 차별화는 경기 회복의 온기가 다른 부문으로 확산되기까지 시간이 소요되기 때문에 발생합니다.



| 질문 | RAG 답변 | 정확성 | 환각 여부 | 비고 |
| --- | --- | --- | --- | --- |
| GDP 성장률 | 2.0% | 높음 | 없음 |  |
| 소비자물가 | 2.0% | 높음 | 없음 |  |
| 수출 전망 | 7952억 달러 | 높음 | 없음 |  |
| 2030년 전망 | 확인 불가 | 정상 | 없음 | 범위 밖 질문 처리 정상 |
| 미국 금리 결정 일정 | 확인 불가 | 정상 | 없음 | 범위 밖 질문 처리 정상 |
| 성장률 반등 | 확인 불가 | 비정상 | 없음 | 맥락 이해 처리 비정상 |
| 부문별 온도차 | K자형 회복국면 | 높음 | 없음 | 맥락 이해 처리 정상 |

## 청킹 전략 비교 실습

이번 실습에서는 동일한 보고서, 동일한 임베딩 모델, 동일한 질문 형식을 유지한 채 청킹 전략만 바꿔 비교합니다.
Pinecone에 다시 업서트하지 않고 로컬 `FAISS` 인덱스를 만들어 빠르게 실험합니다.

| 전략 | 설정 | 기대 효과 |
| --- | --- | --- |
| 전략 A | `chunk_size=200`, `chunk_overlap=20` | 세밀한 검색에 유리하지만 문맥이 잘릴 수 있음 |
| 전략 B | `chunk_size=500`, `chunk_overlap=50` | 검색 정밀도와 문맥 보존의 균형 |
| 전략 C | `chunk_size=1000`, `chunk_overlap=100` | 긴 문맥 유지에 유리하지만 불필요한 정보가 섞일 수 있음 |


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunking_strategies = {
    "전략 A (200/20)": RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=20,
        separators=["\n\n", "\n", " ", ""]
    ),
    "전략 B (500/50)": RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", " ", ""]
    ),
    "전략 C (1000/100)": RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    ),
}

strategy_splits = {}

for name, splitter in chunking_strategies.items():
    current_splits = splitter.split_documents(docs)
    strategy_splits[name] = current_splits

    avg_len = sum(len(doc.page_content) for doc in current_splits) // len(current_splits)
    print(f"{name}: 청크 수 {len(current_splits)}개, 평균 길이 {avg_len}자")


### 로컬 FAISS 인덱스 준비

아래 셀은 각 전략별로 별도의 `FAISS` 벡터스토어와 retriever를 생성합니다.
앞 셀을 건너뛰어도 필요한 청킹 결과가 없으면 자동으로 다시 만듭니다.


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


def format_docs_for_prompt(retrieved_docs):
    return "\n\n".join(
        f"[p.{doc.metadata.get('page', '')}] {doc.page_content}"
        for doc in retrieved_docs
    )


if "chunking_strategies" not in globals():
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    chunking_strategies = {
        "전략 A (200/20)": RecursiveCharacterTextSplitter(
            chunk_size=200,
            chunk_overlap=20,
            separators=["\n\n", "\n", " ", ""]
        ),
        "전략 B (500/50)": RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            separators=["\n\n", "\n", " ", ""]
        ),
        "전략 C (1000/100)": RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=100,
            separators=["\n\n", "\n", " ", ""]
        ),
    }

if "strategy_splits" not in globals():
    strategy_splits = {
        name: splitter.split_documents(docs)
        for name, splitter in chunking_strategies.items()
    }

faiss_vectorstores = {}
faiss_retrievers = {}
faiss_rag_chains = {}

for name, current_splits in strategy_splits.items():
    vectorstore = FAISS.from_documents(current_splits, embedding_model)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    chain = (
        {
            "context": retriever | RunnableLambda(format_docs_for_prompt),
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | parser
    )

    faiss_vectorstores[name] = vectorstore
    faiss_retrievers[name] = retriever
    faiss_rag_chains[name] = chain

print("FAISS 인덱스 준비 완료")
for name, current_splits in strategy_splits.items():
    print(f"- {name}: {len(current_splits)}개 청크")


### 검색 결과 비교

기존과 같은 질문 형식을 유지하면서, 각 전략에서 어떤 청크를 상위로 가져오는지 먼저 확인합니다.

| 확인 항목 | 좋은 경우 | 나쁜 경우 |
| --- | --- | --- |
| 상위 1~3개 청크 | 질문과 직접 관련된 수치와 문장이 포함됨 | 다른 주제 청크가 섞임 |
| 페이지 집중도 | 인접 페이지에 모여 있음 | 무작위 페이지로 분산됨 |
| 문장 완결성 | 핵심 수치와 근거가 함께 보임 | 표나 문장이 중간에서 잘림 |


In [40]:
comparison_questions = globals().get(
    "test_questions",
    [
        "2026년 GDP 성장률 전망치는 얼마인가요?",
        "2026년 소비자물가 상승률 전망치는?",
        "2026년 수출 전망 금액은 얼마인가요?",
        "2030년 경제성장률 전망치는 얼마인가요?",
        "보고서에서 가장 큰 리스크 요인은 무엇인가요?",
    ],
)

for q in comparison_questions:
    print(f"[ Q ] {q}")

    for name, retriever in faiss_retrievers.items():
        print(f"[ {name} 검색 ]")
        results = retriever.invoke(q)

        for i, r in enumerate(results):
            snippet = r.page_content.replace("\n", " ")
            print(f"  {i+1}. (p.{r.metadata.get('page', '')}) {snippet[:120]}")

    print()


[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ 전략 A (200/20) 검색 ]
  1. (p.43) 30    <경제성장 전망1)>  (전년동기대비, %)    2024 2025 2026e) 2027e)  연간 상반 하반 연간 상반 하반 연간 연간  GDP 성장률 2.0 0.3 1.6 1.0 2.4 1.6 2.0 
  2. (p.6) 등으로 당초 예상을 소폭 상회하는 2.2% 상승할 전망이다.    (%, %p) 2024 20251) 2026e)1) 2027e)1)   GDP 성장률 2.0  1.0 ( ― ) 2.0 (+0.2) 1.8 (-0.
  3. (p.79) 66    한국은행 전망에서는 점진적∙보수적인 전망 경향이 나타남   [그림3] 분기별 GDP 성장률1) 및 전망 [그림4] 분기성장률(전년동기대비) 증감과 전망오차2)        주: 1) 속보치 기준      
[ 전략 B (500/50) 검색 ]
  1. (p.43) 30    <경제성장 전망1)>  (전년동기대비, %)    2024 2025 2026e) 2027e)  연간 상반 하반 연간 상반 하반 연간 연간  GDP 성장률 2.0 0.3 1.6 1.0 2.4 1.6 2.0 
  2. (p.9) ▪ 27년에는 내수 회복세가 지속되는 가운데 수출도 세계경제 성장세 지속, 반도체  공급능력 확충 등으로 증가하며 1.8%의 견조한 성장세를 나타낼 전망이다.        <국내 성장률 전망1)> <국내 GDP 전망
  3. (p.79) 66    한국은행 전망에서는 점진적∙보수적인 전망 경향이 나타남   [그림3] 분기별 GDP 성장률1) 및 전망 [그림4] 분기성장률(전년동기대비) 증감과 전망오차2)        주: 1) 속보치 기준      
[ 전략 C (1000/100) 검색 ]
  1. (p.9) < 요약 4/8 >  6       경제전망     금년 성장률은 美관세 영향과 건설투자의 더딘 회복에도 불구하고 반도체 경기 개선세  확대, 예상보다 양호한 세계경제 흐

### 답변 비교

이제 같은 질문을 전략별 `RAG` 체인에 넣어 답변 차이를 비교합니다.
평가 포인트는 정확한 수치 포함 여부, 문서 밖 정보 생성 여부, 답변의 간결성입니다.


In [41]:
for q in comparison_questions:
    print(f"[ Q ] {q}")

    for name, chain in faiss_rag_chains.items():
        answer = chain.invoke(q)
        print(f"[ {name} ] {answer}")

    print()


[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ 전략 A (200/20) ] 2026년 GDP 성장률 전망치는 2.0%입니다.
[ 전략 B (500/50) ] 2026년 GDP 성장률 전망치는 2.0%입니다.
[ 전략 C (1000/100) ] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 2026년 소비자물가 상승률 전망치는?
[ 전략 A (200/20) ] 2026년 소비자물가 상승률 전망치는 2.2%입니다.
[ 전략 B (500/50) ] 2026년 소비자물가 상승률 전망치는 2.0%입니다.
[ 전략 C (1000/100) ] 2026년 소비자물가 상승률 전망치는 2.0%입니다.

[ Q ] 2026년 수출 전망 금액은 얼마인가요?
[ 전략 A (200/20) ] 보고서에서 확인되지 않습니다.
[ 전략 B (500/50) ] 2026년 수출 전망 금액은 7,952억 달러입니다.
[ 전략 C (1000/100) ] 2026년 수출 전망 금액은 7,952억 달러입니다.

[ Q ] 2030년 경제성장률 전망은?
[ 전략 A (200/20) ] 보고서에서 확인되지 않습니다.
[ 전략 B (500/50) ] 보고서에서 확인되지 않습니다.
[ 전략 C (1000/100) ] 보고서에서 확인되지 않습니다.

[ Q ] 미국 연방준비제도의 금리 결정 일정은?
[ 전략 A (200/20) ] 보고서에서 확인되지 않습니다.
[ 전략 B (500/50) ] 보고서에서 확인되지 않습니다.
[ 전략 C (1000/100) ] 보고서에서 확인되지 않습니다.

[ Q ] 성장률이 반등하는 주요 원인은?
[ 전략 A (200/20) ] 보고서에서 확인되지 않습니다.
[ 전략 B (500/50) ] 보고서에서 확인되지 않습니다.
[ 전략 C (1000/100) ] 보고서에서 확인되지 않습니다.

[ Q ] 부문별 온도차가 발생하는 이유는?
[ 전략 A (200/20) ] 부문별 온도차는 부문별 성장 차별화가 심화됨에 따라 발생합니다. 같은 정도의 성장을 

### 전략별 자동 요약

아래 셀은 질문별 답변을 간단한 규칙으로 점검해 전략별 특징을 자동으로 요약합니다.
정밀한 평가는 아니지만, 어떤 전략이 상대적으로 안정적인지 빠르게 훑어보기에 유용합니다.


In [42]:
from collections import defaultdict


def score_answer(answer):
    score = 0
    reasons = []

    if any(ch.isdigit() for ch in answer):
        score += 2
        reasons.append("수치 포함")

    if "보고서에서 확인되지 않습니다" in answer:
        score += 1
        reasons.append("범위 밖 질문에 신중하게 응답")

    if len(answer) <= 120:
        score += 1
        reasons.append("답변이 간결함")

    if any(keyword in answer for keyword in ["약", "전망", "%", "억", "조", "분기"]):
        score += 1
        reasons.append("전망 정보 표현 포함")

    note = ", ".join(reasons) if reasons else "특징 없음"
    return score, note


strategy_summary = defaultdict(list)
strategy_total_scores = defaultdict(int)

for q in comparison_questions:
    print(f"[ Q ] {q}")

    best_name = None
    best_score = -1

    for name, chain in faiss_rag_chains.items():
        answer = chain.invoke(q)
        score, note = score_answer(answer)
        strategy_summary[name].append((q, score, note, answer))
        strategy_total_scores[name] += score

        print(f"- {name}: {score}점 | {note}")

        if score > best_score:
            best_score = score
            best_name = name

    print(f"  => 자동 판정 우세 전략: {best_name}")
    print()

print("[ 전략별 총점 ]")
for name, total in sorted(strategy_total_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"- {name}: {total}점")


[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
- 전략 A (200/20): 4점 | 수치 포함, 답변이 간결함, 전망 정보 표현 포함
- 전략 B (500/50): 4점 | 수치 포함, 답변이 간결함, 전망 정보 표현 포함
- 전략 C (1000/100): 4점 | 수치 포함, 답변이 간결함, 전망 정보 표현 포함
  => 자동 판정 우세 전략: 전략 A (200/20)

[ Q ] 2026년 소비자물가 상승률 전망치는?
- 전략 A (200/20): 4점 | 수치 포함, 답변이 간결함, 전망 정보 표현 포함
- 전략 B (500/50): 4점 | 수치 포함, 답변이 간결함, 전망 정보 표현 포함
- 전략 C (1000/100): 4점 | 수치 포함, 답변이 간결함, 전망 정보 표현 포함
  => 자동 판정 우세 전략: 전략 A (200/20)

[ Q ] 2026년 수출 전망 금액은 얼마인가요?
- 전략 A (200/20): 2점 | 범위 밖 질문에 신중하게 응답, 답변이 간결함
- 전략 B (500/50): 4점 | 수치 포함, 답변이 간결함, 전망 정보 표현 포함
- 전략 C (1000/100): 4점 | 수치 포함, 답변이 간결함, 전망 정보 표현 포함
  => 자동 판정 우세 전략: 전략 B (500/50)

[ Q ] 2030년 경제성장률 전망은?
- 전략 A (200/20): 2점 | 범위 밖 질문에 신중하게 응답, 답변이 간결함
- 전략 B (500/50): 2점 | 범위 밖 질문에 신중하게 응답, 답변이 간결함
- 전략 C (1000/100): 2점 | 범위 밖 질문에 신중하게 응답, 답변이 간결함
  => 자동 판정 우세 전략: 전략 A (200/20)

[ Q ] 미국 연방준비제도의 금리 결정 일정은?
- 전략 A (200/20): 2점 | 범위 밖 질문에 신중하게 응답, 답변이 간결함
- 전략 B (500/50): 2점 | 범위 밖 질문에 신중하게 응답, 답변이 간결함
- 전략 C (1000/100): 2점 | 범위 밖 질문에

### 비교 결과 정리표

아래 표는 답변 비교 셀과 전략별 자동 요약 셀의 결과를 함께 반영해 정리한 것입니다.
자동 점수 총합은 `전략 C(19점) > 전략 B(18점) > 전략 A(17점)`이었지만, 실제 답변의 안정성과 검색 일관성까지 함께 보면 `전략 B`가 가장 균형적인 선택으로 보입니다.

| 질문 | 전략 A | 전략 B | 전략 C | 가장 적합한 전략 | 판단 근거 |
| --- | --- | --- | --- | --- | --- |
| 2026년 GDP 성장률 전망치는 얼마인가요? | `2.0%`로 정확 | `2.0%`로 정확 | `2.0%`로 정확 | 공동 우수 | 세 전략 모두 같은 정답을 간결하게 답변함 |
| 2026년 소비자물가 상승률 전망치는? | `2.2%`로 답변 | `2.0%`로 답변 | `2.0%`로 답변 | 전략 B | 기존 실습 결과와 검색 문맥상 `2.0%`가 더 적절해 보이며, B가 더 직접적인 근거 청크를 가져옴 |
| 2026년 수출 전망 금액은 얼마인가요? | 확인 불가로 응답 | `7,952억 달러`로 정확 | `7,952억 달러`로 정확 | 전략 B | A는 필요한 수치를 놓쳤고, B와 C는 맞췄지만 B가 더 안정적으로 검색했음 |
| 2030년 경제성장률 전망은? | 확인 불가 | 확인 불가 | 확인 불가 | 공동 우수 | 보고서 범위를 벗어난 질문에 세 전략 모두 환각 없이 응답함 |
| 미국 연방준비제도의 금리 결정 일정은? | 확인 불가 | 확인 불가 | 확인 불가 | 공동 우수 | 보고서 외부 정보에 대해 세 전략 모두 보수적으로 응답함 |
| 성장률이 반등하는 주요 원인은? | 확인 불가 | 확인 불가 | 확인 불가 | 공동 보통 | 현재 질문이 추상적이라 세 전략 모두 충분한 근거를 끌어오지 못함 |
| 부문별 온도차가 발생하는 이유는? | 짧지만 다소 포괄적 | 구체적이고 균형적 | 길고 확장된 설명 | 전략 B | B가 핵심 원인을 가장 안정적으로 설명했고, C는 다소 과설명 경향이 있었음 |

| 종합 항목 | 결과 |
| --- | --- |
| 자동 요약 총점 | 전략 C 19점, 전략 B 18점, 전략 A 17점 |
| 실무용 추천 | 전략 B (`chunk_size=500`, `chunk_overlap=50`) |
| 추천 이유 | 수치 질문에서는 정확했고, 서술형 질문에서는 과도하게 길지 않으면서도 필요한 문맥을 가장 안정적으로 유지함 |
| 전략 A 특징 | 짧고 보수적이지만, 수출 전망처럼 필요한 수치를 놓치는 경우가 있었음 |
| 전략 C 특징 | 총점은 가장 높았지만, 일부 질문에서 문맥이 길어져 답변이 과하게 확장되는 경향이 있었음 |
